In [ ]:
# 1. Configure this notebook before using Run All (Google Colab or local Jupyter).
# Current PyPI release is 0.1.1; these v0.2.0 examples default to a pinned Git commit.
NSB_SOURCE = "github"  # "github" or "pypi"
NSB_VERSION = "0.2.0"  # exact nsb-py release in PyPI mode (must be published)
NSB_GIT_REF = "3b0832418a91576b0b3c20fec1f31c0446761059"  # exact VPRamon/NSB commit for GitHub mode
INSTALL_DEPENDENCIES = True  # False only for an already prepared, verified kernel


# Continuum airglow at fixed geometry

This example uses the public NSB Python API to vary explicit F10.7 solar radio
flux at one fixed Paranal observer, ICRS direction and UTC instant. It integrates
the PALACE continuum over 300–650 nm and produces one figure. No external data
are required: NSB uses its bundled, manifest-verified scientific assets.

This example targets the v0.2.0 public API with the explicit
PALACE continuum model. For a publication, record the exact released version
or source commit and environment. This demonstration is not an absolute
observational validation.

Run the first configuration cell and then **Run All** in a fresh Python kernel,
including Google Colab. The setup cell installs `nsb-py` (the distribution that
provides `import nsb`) and Matplotlib from public sources. In PyPI mode choose
an actual published version supporting this API; in GitHub mode use a full commit
SHA and no local checkout. GitHub source builds need Rust 1.89+ (automatically
installed on Linux when missing); the initial installation requires network
access. The scientific calculation itself uses bundled assets and no downloads.


In [ ]:
# Install from PyPI or a remote GitHub revision (works without a local checkout).
import importlib.metadata as _metadata
import json as _json
import os as _os
from pathlib import Path as _Path
import re as _re
import shutil as _shutil
import subprocess as _subprocess
import sys as _sys
import tempfile as _tempfile
from urllib.request import urlretrieve as _urlretrieve

if NSB_SOURCE not in ("pypi", "github"):
    raise ValueError("NSB_SOURCE must be 'pypi' or 'github'")
if "nsb" in _sys.modules:
    raise RuntimeError("Restart the kernel before running setup again or changing NSB_SOURCE")

if NSB_SOURCE == "github":
    if not _re.fullmatch(r"[0-9a-f]{40}", NSB_GIT_REF):
        raise ValueError("Set NSB_GIT_REF to a full 40-character commit SHA for reproducibility")
    if INSTALL_DEPENDENCIES:
        _rust = _shutil.which("rustc")
        _rust_info = _subprocess.run([_rust, "--version"], capture_output=True, text=True).stdout if _rust else ""
        _match = _re.search(r"rustc (\d+)\.(\d+)", _rust_info)
        if not _match or tuple(map(int, _match.groups())) < (1, 89):
            if _sys.platform != "linux":
                raise RuntimeError("Building NSB from GitHub requires Rust 1.89+; install rustup first")
            print("Installing/updating the official Rust toolchain for the source build...")
            if _shutil.which("rustup"):
                _subprocess.run(["rustup", "default", "stable"], check=True)
                _subprocess.run(["rustup", "update", "stable"], check=True)
            else:
                with _tempfile.TemporaryDirectory() as _tmp:
                    _script = _Path(_tmp) / "rustup-init.sh"
                    _urlretrieve("https://sh.rustup.rs", _script)
                    _subprocess.run(["sh", str(_script), "-y", "--profile", "minimal"], check=True)
            _os.environ["PATH"] = str(_Path.home() / ".cargo/bin") + _os.pathsep + _os.environ["PATH"]
    _package = f"nsb-py @ git+https://github.com/VPRamon/NSB.git@{NSB_GIT_REF}"
else:
    _package = f"nsb-py=={NSB_VERSION}"

if INSTALL_DEPENDENCIES:
    _subprocess.run(
        [_sys.executable, "-m", "pip", "install", "--quiet", _package, "matplotlib"],
        check=True,
    )
else:
    print("Package installation skipped (INSTALL_DEPENDENCIES=False).")

_dist = _metadata.distribution("nsb-py")
if NSB_SOURCE == "pypi" and _dist.version != NSB_VERSION:
    raise RuntimeError(f"Expected nsb-py=={NSB_VERSION}, found {_dist.version}")
_direct = _json.loads(_dist.read_text("direct_url.json") or "{}")
if NSB_SOURCE == "github":
    if (_direct.get("url") != "https://github.com/VPRamon/NSB.git"
            or _direct.get("vcs_info", {}).get("commit_id") != NSB_GIT_REF):
        raise RuntimeError("The installed nsb-py is not from the selected NSB GitHub commit")
elif _direct.get("vcs_info"):
    raise RuntimeError("Expected a PyPI release, but nsb-py came from Git; restart and reinstall")

import nsb as _nsb
_files = {_Path(_dist.locate_file(p)).resolve() for p in (_dist.files or [])}
if _Path(_nsb.__file__).resolve() not in _files:
    raise RuntimeError("Import 'nsb' is shadowed by a different package; inspect/uninstall the conflicting distribution")
if not hasattr(_nsb, "NsbEvaluator") or not hasattr(_nsb, "NsbModelConfig"):
    raise RuntimeError("The installed 'nsb' does not expose the official NSB Python API")
print(f"NSB ready: nsb-py {_dist.version}, source={NSB_SOURCE}"
      + (f", commit={NSB_GIT_REF}" if NSB_SOURCE == "github" else ""))


In [ ]:
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import nsb
import siderust

observer = siderust.Observer(-70.4045, -24.6272, 2635.0)  # east-positive degrees, metres
query = nsb.PointQuery(
    observer, datetime(2023, 9, 15, 5, tzinfo=timezone.utc),
    siderust.Direction(10.0, -20.0),  # ICRS RA, Dec in degrees
    components=nsb.ComponentMask.AIRGLOW,
)
solar_flux = [67.0, 80.0, 100.0, 120.0, 140.0, 166.0]  # sfu
radiance = []
for sfu in solar_flux:
    config = (nsb.NsbModelConfig.generic_clear_sky()
              .with_airglow_model(nsb.AirglowModel.PARANAL_PALACE_V1)
              .with_solar_radio_flux_sfu(sfu))
    evaluator = nsb.NsbEvaluator(config)
    radiance.append(evaluator.integrate_band(query, 300.0, 650.0))
assert all(b > a > 0 for a, b in zip(radiance, radiance[1:]))


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.2), layout="constrained")
ax.plot(solar_flux, radiance, "o-")
ax.set(xlabel="F10.7 solar radio flux (sfu)",
       ylabel="Photon radiance (photons cm⁻² ns⁻¹ sr⁻¹)",
       title="PALACE continuum, 300–650 nm, fixed Paranal geometry")
ax.grid(alpha=0.2)
plt.show()


## Interpretation

The ordinate is band-integrated photon radiance, not detector counts or
photometric magnitude. The explicit solar inputs span 67–166 sfu; all other
inputs stay fixed. The continuum rises for this configuration. This planning
climatology is derived from Paranal and does not certify absolute accuracy at
other sites or on an individual night. Oxygen lines, starlight, zodiacal light
and moonlight are excluded; unknown observational uncertainty is not zero.

The figure demonstrates public API use and deterministic parameter control.
For scope and evidence, consult the
[scientific validation guide](../docs/scientific-validation.md), NSB's
[airglow guide](../docs/nsb_components/airglow/README.md) and
[restricted-band contract](../docs/specifications/restricted-band-radiance.md)
for model provenance and domain limits.
